# Building the two MICrONS activity files

Every figure in this repo runs off the structural data alone, except for the panels of
figures 6, 7, S14 and S15 that read the MICrONS *functional* recordings. Those live in
two HDF5 files, and this notebook is what makes them:

| file | size | what it holds |
|---|---|---|
| `coreg_manual_v4_calcium_v2.h5` | ~19 GB | per-unit calcium and deconvolved spike traces, keyed by `nucleus_id`, with each unit's `ms_delay` and each scan's real `frame_times` |
| `microns_per_scan_stimuli.h5`   | ~68 MB | per-scan trial table (the oracle clip windows) and condition metadata |

**Neither file is in the Zenodo snapshot.** At 19 GB the calcium file is far too large to
redistribute, and the stimulus file is built in the same pass, so both are made here. This
is the only route to them: `download_data_cave.py` cannot produce them either.

Both come out of the MICrONS `microns_phase3_nda` DataJoint database, which is only
reachable from inside the project's database container — hence the round trip below.

---

## Before you start

You need `coregistration_manual_v4.csv`. Unlike the two H5 files, this one *is* shipped —
either route to the structural data puts it at `data/raw_tables/coregistration_manual_v4.csv`:

```
python download_data_zenodo.py              # the Zenodo snapshot
python download_data_cave.py --steps raw    # or CAVE — a few minutes, needs an account
```

Take whichever you already ran; the file is the same. All the container needs is a copy of
it sitting next to this notebook (step 2 below).

---

## Step by step

**1. Get the container running.** Follow the upstream instructions at
[cajal/microns-nda-access](https://github.com/cajal/microns-nda-access?tab=readme-ov-file#database-container)
— clone that repo, put your DataJoint credentials where its README says, and bring the
stack up. It exposes a JupyterLab server with `microns_phase3` already installed and the
database reachable.

**2. Copy two files into the container's notebook directory**, alongside each other:

- this notebook, and
- `coregistration_manual_v4.csv` (from `data/raw_tables/` — see above)

**3. Run this notebook top to bottom.** Section 1 writes the calcium file, section 2 the
stimuli file. Section 1 probes a single unit and then does a 20-nucleus dry run before the
full pass — read the output of both before letting the full run go; it takes hours and
writes ~19 GB.

**4. Copy the two `.h5` files back out** of the container and into `data/activity/` in
this repo. The figure notebooks look for them there and say so if they are missing.

**5. Back in this repo, run the ensemble pipeline:**

```
python scripts/ensemble_run.py
```

It reads both files and writes the pickles that figures 6, S14 and S15 plot into
`data/activity/ensembles/`. Figure 7 and figure 6 panel A read the calcium file directly
and need nothing further.

---

## Section 1 — calcium (`coreg_manual_v4_calcium_v2.h5`)


Adapted from the MICrONS tutorial
[`Matched_Cell_Functional_Data.ipynb`](https://github.com/cajal/microns_phase3_nda/blob/v8/tutorial_notebooks/Matched_Cell_Functional_Data.ipynb),
trimmed to the extraction path and extended to record `ms_delay` and the real per-scan
`frame_times` (see section 1).

In [ ]:
import h5py
import numpy as np
import pandas as pd
from collections import Counter
from h5py import special_dtype
from microns_phase3 import nda, utils
from tqdm.notebook import tqdm

In [ ]:
# Put coregistration_manual_v4.csv next to this notebook — the top cell says where to get it.
co_df = pd.read_csv('coregistration_manual_v4.csv', index_col=0)

### v2 extraction — what changed and why

Two fixes, both discovered while chasing a spurious latency signal in `axon_marom_project/signal_delay.ipynb`.

**1. Missing acquisition timing (`ms_delay`).** v1 fetched only traces + `nframes`/`fps`, so every unit was implicitly assumed to be sampled at `arange(nframes)/fps`. That is false for resonant/mesoscope scanning: `nda.ScanUnit.ms_delay` is *"delay from start of frame (field 1 pixel 1) to recording of this unit (milliseconds)"* and spans 0–159 ms at 6.3 Hz — up to a full frame. It is spatially structured (depth / field order), so it biases any pairwise-latency measurement in a way that correlates with anatomy. v2 fetches `ms_delay`, `um_x/y/z`, `px_x/y`, `field`, `mask_id`, `field_x/y/z`, `mask_type`, `nfields`.

**2. Real frame times.** `nda.ScanTimes.frame_times` are the true acquisition times; `arange(nframes)/fps` drifts from them by ~2.5 s over a 40k-frame scan (nominal `fps=6.3009` vs true `6.2984`). Stored per scan under `/scans/`. Only 16 scans, ~5 MB.

**3. Keyed by `nucleus_id`, not `pt_root_id`.** `pt_root_id` is materialization-specific (v1507 here), which forced a v1507↔v1718 bridge on every consumer and caused repeated bugs. `nucleus_id` (`co_df.target_id`) is stable. On this table `target_id → pt_root_id` is 1:1 for all 15,439 nuclei, but **one `pt_root_id` maps to 6 distinct nuclei** — a merge-error segment that v1 silently collapsed into a single group.

Also: v1's bare `except: return None` swallowed every failure. v2 records and reports the reasons.

Output file is `coreg_manual_v4_calcium_v2.h5` — written alongside v1, not over it.

In [ ]:
"""Per-unit fetch — v2.

Changes vs. v1 (see the markdown cell above for why):
  * adds ScanUnit.ms_delay + um_x/y/z + px_x/y + field/mask_id, and Field.field_x/y/z.
    ms_delay is the per-unit acquisition offset within a frame (0..1/fps, i.e. 0..159 ms
    at 6.3 Hz). Without it every unit is implicitly assumed to be sampled at
    arange(nframes)/fps, which is wrong by up to a full frame and is spatially
    structured (depth / field order) -> a confound for any latency analysis.
  * adds MaskClassification.mask_type.
  * no longer swallows every exception. v1's bare `except: return None` silently
    dropped units; now the reason is printed and counted.

frame_times is per (session, scan_idx), not per unit -> stored once in a top-level
`scans` group by save_neurons_to_h5, not repeated 19k times.
"""

# ScanUnit carries `field` + `mask_id` as secondary attributes via its FK to
# Fluorescence (PK: session, scan_idx, field, mask_id).
SCAN_UNIT_ATTRS = ['ms_delay', 'um_x', 'um_y', 'um_z', 'px_x', 'px_y', 'field', 'mask_id']

_fetch_failures = Counter()


def fetch_unit_data(unit_key, strict=False):
    """Fetch traces + acquisition metadata for one unit.

    Returns a flat dict of h5-writable values, or None if the unit could not be
    fetched (reason recorded in `_fetch_failures`). Set strict=True to raise instead.
    """
    try:
        oracle_traces, oracle_score = utils.fetch_oracle_raster(unit_key)
        nframes, fps, nfields = (nda.Scan & unit_key).fetch1('nframes', 'fps', 'nfields')
        spike_trace = (nda.Activity & unit_key).fetch1('trace')
        calcium_trace = (nda.ScanUnit * nda.Fluorescence & unit_key).fetch1('trace')
        pupil_radius = (nda.ManualPupil & unit_key).fetch1('pupil_maj_r')
        treadmill = (nda.Treadmill & unit_key).fetch1('treadmill_velocity')

        # --- acquisition geometry / timing (the v1 omission) ---
        su = (nda.ScanUnit & unit_key).fetch1(*SCAN_UNIT_ATTRS)
        unit_meta = dict(zip(SCAN_UNIT_ATTRS, su))

        field_key = {'session': unit_key['session'], 'scan_idx': unit_key['scan_idx'],
                     'field': unit_meta['field']}
        field_x, field_y, field_z = (nda.Field & field_key).fetch1('field_x', 'field_y', 'field_z')

        # NB: MaskClassification's PK is (session, scan_idx, field, mask_id) — it has no
        # unit_id, and DataJoint *silently ignores* restriction attributes a table
        # doesn't have. `nda.MaskClassification & unit_key` would therefore match every
        # mask in the scan and fetch1 would raise. Join through ScanUnit (which carries
        # field + mask_id) so the restriction actually resolves to one row.
        try:
            mask_type = (nda.ScanUnit * nda.MaskClassification & unit_key).fetch1('mask_type')
        except Exception:
            mask_type = ''   # optional — absent for some units

        out = {
            'nframes': nframes,
            'fps': fps,
            'nfields': nfields,
            'spike_trace': spike_trace,
            'calcium_trace': calcium_trace,
            'pupil_radius': pupil_radius,
            'treadmill': treadmill,
            'session': unit_key['session'],
            'scan_idx': unit_key['scan_idx'],
            'unit_id': unit_key['unit_id'],
            'oracle_score': oracle_score,
            'field_x': field_x,
            'field_y': field_y,
            'field_z': field_z,
            'mask_type': mask_type,
        }
        out.update(unit_meta)
        return out

    except Exception as e:
        if strict:
            raise
        _fetch_failures[f'{type(e).__name__}: {e}'] += 1
        return None


def fetch_scan_times(scan_key):
    """frame_times (field 1 of the scan) + ndepths, from nda.ScanTimes.

    frame_times are the *real* acquisition times. arange(nframes)/fps drifts from
    them by ~2.5 s over a 40k-frame scan, so anything aligning traces to stimulus
    trial times must use these, not the nominal fps.
    """
    frame_times, ndepths = (nda.ScanTimes & scan_key).fetch1('frame_times', 'ndepths')
    return np.asarray(frame_times, dtype=np.float64), int(ndepths)


In [ ]:
"""H5 writer — v2. Keyed by NUCLEUS ID, not pt_root_id.

Why: pt_root_id is materialization-specific (this table is v1507), so every
downstream consumer needed a v1507<->v1718 bridge, and that bridge has been the
source of several bugs. nucleus_id (= co_df.target_id) is stable across
materializations, so keying by it removes the bridge entirely.

It is also strictly more correct on this table:
  * target_id -> pt_root_id is 1:1 for all 15,439 nuclei
  * but one pt_root_id maps to SIX distinct nuclei (a merge-error segment).
    Keying by root_id collapses those six cells into one group; keying by
    nucleus keeps them separate.
  * 15,439 nucleus groups vs 15,434 root_id groups.

Layout:
    /neurons/<nucleus_id>/unit_<i>/{calcium_trace, spike_trace, ..., ms_delay,
                                    um_x, um_y, um_z, px_x, px_y, field, mask_id,
                                    field_x, field_y, field_z, mask_type,
                                    pt_root_id_v1507}
    /scans/s<session:02d>/scan<scan_idx:02d>/{frame_times, ndepths}
Attrs on /: nucleus_key_version, source_table.
"""

def save_neurons_to_h5(filename, co_df, nucleus_col='target_id', limit=None):
    vlen_str = h5py.special_dtype(vlen=str)
    _fetch_failures.clear()

    nucleus_ids = sorted(co_df[nucleus_col].unique())
    if limit:
        nucleus_ids = nucleus_ids[:limit]

    n_units = n_skipped = 0
    with h5py.File(filename, 'w') as f:
        f.attrs['key'] = 'nucleus_id'
        f.attrs['source_table'] = 'coregistration_manual_v4'
        f.attrs['note'] = ('neurons keyed by nucleus_id (co_df.target_id), stable across '
                           'materializations. pt_root_id_v1507 kept per unit for traceability.')

        # --- per-scan acquisition times (16 scans, ~5 MB total) ---
        scans_group = f.create_group('scans')
        for (s, sc) in sorted(co_df.groupby(['session', 'scan_idx']).groups):
            try:
                frame_times, ndepths = fetch_scan_times({'session': int(s), 'scan_idx': int(sc)})
            except Exception as e:
                print(f'  scan times failed for ({s},{sc}): {e}')
                continue
            g = scans_group.require_group(f's{int(s):02d}').require_group(f'scan{int(sc):02d}')
            g.create_dataset('frame_times', data=frame_times)
            g.create_dataset('ndepths', data=ndepths)
        print(f'saved frame_times for {len(list(scans_group))} sessions')

        # --- per-nucleus units ---
        neurons_group = f.create_group('neurons')
        for nucleus_id in tqdm(nucleus_ids, desc='nuclei'):
            entries = co_df[co_df[nucleus_col] == nucleus_id]
            if entries.empty:
                continue

            neuron_group = neurons_group.create_group(f'{int(nucleus_id)}')
            written = 0
            for _, entry in entries.iterrows():
                unit_key = {k: int(entry[k]) for k in ('session', 'scan_idx', 'unit_id')}
                unit_data = fetch_unit_data(unit_key)
                if not unit_data:
                    n_skipped += 1
                    continue
                unit_data['pt_root_id_v1507'] = int(entry['pt_root_id'])
                unit_data['nucleus_id'] = int(nucleus_id)

                unit_group = neuron_group.create_group(f'unit_{written}')
                for key, value in unit_data.items():
                    if isinstance(value, str):
                        unit_group.create_dataset(key, data=np.array(value, dtype=object),
                                                  dtype=vlen_str)
                    else:
                        unit_group.create_dataset(key, data=value)
                written += 1
                n_units += 1

            if written == 0:       # don't leave empty nucleus groups behind
                del neurons_group[f'{int(nucleus_id)}']

    print(f'\nnuclei written: {len(list(h5py.File(filename, "r")["neurons"]))}, '
          f'units: {n_units}, units skipped: {n_skipped}')
    if _fetch_failures:
        print('\nfailure reasons (top 10):')
        for reason, count in _fetch_failures.most_common(10):
            print(f'  {count:>6}  {reason[:120]}')


In [ ]:
# PROBE ONE UNIT — run this first. strict=True so schema mistakes raise
# loudly instead of being counted as a failure. Confirms every new attribute
# actually exists and that ms_delay looks like a within-frame offset.
_probe = co_df.iloc[0]
_key = {k: int(_probe[k]) for k in ('session', 'scan_idx', 'unit_id')}
_d = fetch_unit_data(_key, strict=True)

for k, v in _d.items():
    print(f'  {k:20s} {np.shape(v)}  {v if np.ndim(v) == 0 else ""}')

frame_period_ms = 1000.0 / _d['fps']
print(f'\nframe period = {frame_period_ms:.1f} ms   ms_delay = {_d["ms_delay"]} ms')
assert 0 <= _d['ms_delay'] <= frame_period_ms + 1, 'ms_delay outside one frame — check units'

_ft, _nd = fetch_scan_times({'session': _key['session'], 'scan_idx': _key['scan_idx']})
_drift = np.abs(_ft - _ft[0] - np.arange(len(_ft)) / _d['fps']).max()
print(f'frame_times: n={len(_ft)}, ndepths={_nd}, '
      f'max drift vs arange/fps = {_drift * 1000:.0f} ms')

In [ ]:
# Dry run on 20 nuclei — check the layout and the failure report before the full pass.
save_neurons_to_h5('_dry_run_calcium.h5', co_df, limit=20)

with h5py.File('_dry_run_calcium.h5', 'r') as f:
    print('\nroot attrs:', dict(f.attrs))
    print('scans:', {s: list(f['scans'][s]) for s in f['scans']})
    n0 = list(f['neurons'])[0]
    u0 = list(f['neurons'][n0])[0]
    print(f'\n/neurons/{n0}/{u0} contains:\n  ', sorted(f['neurons'][n0][u0].keys()))

In [ ]:
# FULL RUN — ~19k units. Only after the two cells above look right.
save_neurons_to_h5('coreg_manual_v4_calcium_v2.h5', co_df)

---

## Section 2 — stimuli (`microns_per_scan_stimuli.h5`)

Per-scan trial tables: which stimulus was shown over which frame range, plus the condition
metadata behind each `condition_hash`. Figure 6 panel A reads this to find the oracle clip
windows (`activity_utils.load_scan_trials_df` / `get_oracle_condition_hashes`).

Small and quick — minutes, not hours. Like the calcium file, it is not in the Zenodo
snapshot.


In [ ]:
import numpy as np, pandas as pd, h5py, json
from decimal import Decimal

def _stim_kind(stim_type: str) -> str:
    # e.g. 'stimulus.clip' -> 'clip'
    return (stim_type or "").split(".")[-1].lower()

def _to_jsonable(x):
    """Convert numpy/Decimal/bytes recursively into JSON-safe python."""
    import numpy as _np
    from collections.abc import Mapping, Sequence
    if isinstance(x, Mapping):
        return {str(k): _to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple, set)):
        return [_to_jsonable(v) for v in x]
    if isinstance(x, _np.ndarray):
        if x.dtype.kind in ('S','U','O'):
            return x.astype(str).tolist()
        return x.tolist()
    if isinstance(x, (_np.integer,)):  return int(x)
    if isinstance(x, (_np.floating,)): return float(x)
    if isinstance(x, (_np.bool_,)):    return bool(x)
    if isinstance(x, Decimal):         return float(x)
    if isinstance(x, (bytes, _np.bytes_)): return x.decode('utf-8', 'replace')
    return x

def _condition_info(cond_hash: str, stim_type: str):
    """Look up per-condition metadata in nda.Clip/Monet2/Trippy; robust to padded hashes."""
    ch = (str(cond_hash) or "").strip()
    kind = _stim_kind(stim_type)
    table = {'clip': nda.Clip, 'monet2': nda.Monet2, 'trippy': nda.Trippy}.get(kind)
    if not ch or table is None:
        return {}
    try:
        row = (table & {'condition_hash': ch}).fetch1()
        # Drop heavy binary payloads if present
        for big in ('movie', 'clip', 'packed_phase_movie'):
            if big in row:
                row.pop(big)
        return row
    except Exception:
        return {}


def fetch_scan_trials(scan_key):
    """
    scan_key = {'session': int, 'scan_idx': int}
    Returns:
      frame_times : (nframes,) float array (sec)
      trials_df   : tidy df with columns:
                    ['trial_idx','type','start_idx','end_idx','start_time','end_time','condition_hash']
      conds_df    : one row per unique condition_hash in this scan, with 'type' and condition_info fields
    """
    frame_times = (nda.ScanTimes & scan_key).fetch1('frame_times')

    rows = (nda.Trial & scan_key).fetch(
        'trial_idx','type','start_idx','end_idx',
        'start_frame_time','end_frame_time','condition_hash',
        order_by='trial_idx', as_dict=True
    )
    trials_df = pd.DataFrame(rows).rename(columns={
        'start_frame_time':'start_time', 'end_frame_time':'end_time'
    })

    if trials_df.empty:
        return np.asarray(frame_times), trials_df, pd.DataFrame()

    # unique conditions for this scan
    uniq = trials_df[['type','condition_hash']].copy()
    uniq['condition_hash'] = uniq['condition_hash'].astype(str).str.strip()
    uniq = uniq.drop_duplicates()

    records = []
    for r in uniq.itertuples(index=False):
        info = _condition_info(r.condition_hash, r.type)
        rec = {
            'session': scan_key['session'],
            'scan_idx': scan_key['scan_idx'],
            'condition_hash': r.condition_hash,
            'type': r.type,
        }
        rec.update(_to_jsonable(info))
        records.append(rec)
    conds_df = pd.DataFrame.from_records(records)

    return np.asarray(frame_times), trials_df, conds_df


def save_scans_to_h5(filename, scan_keys):
    """
    scan_keys: iterable of {'session':int,'scan_idx':int}
    Creates a NEW file (mode='w') with per-scan trials & condition metadata.
    """
    vlen_str = h5py.special_dtype(vlen=str)
    with h5py.File(filename, 'w') as f:
        root = f.create_group('scans')
        seen_scans = set()
        n_scans = n_conds = 0

        for sk in tqdm(scan_keys):
            s, sc = int(sk['session']), int(sk['scan_idx'])
            key = (s, sc)
            if key in seen_scans:
                continue
            seen_scans.add(key)

            frame_times, trials_df, conds_df = fetch_scan_trials({'session': s, 'scan_idx': sc})
            g_sess = root.require_group(f"s{s:02d}")
            g_scan = g_sess.require_group(f"scan{sc:02d}")

            # frame_times
            if 'frame_times' in g_scan:
                del g_scan['frame_times']
            g_scan.create_dataset('frame_times', data=np.asarray(frame_times, dtype=np.float64))

            # trials
            if 'trials' in g_scan:
                del g_scan['trials']
            tg = g_scan.create_group('trials')
            if not trials_df.empty:
                tg.create_dataset('trial_idx',      data=trials_df['trial_idx'].to_numpy(np.int32))
                tg.create_dataset('type',           data=trials_df['type'].astype(str).to_numpy(dtype=object), dtype=vlen_str)
                tg.create_dataset('start_idx',      data=trials_df['start_idx'].to_numpy(np.int32))
                tg.create_dataset('end_idx',        data=trials_df['end_idx'].to_numpy(np.int32))
                tg.create_dataset('start_time',     data=trials_df['start_time'].to_numpy(np.float64))
                tg.create_dataset('end_time',       data=trials_df['end_time'].to_numpy(np.float64))
                tg.create_dataset('condition_hash', data=trials_df['condition_hash'].astype(str).str.strip().to_numpy(dtype=object), dtype=vlen_str)

            # conditions
            if 'conditions' in g_scan:
                del g_scan['conditions']
            cg = g_scan.create_group('conditions')
            if not conds_df.empty:
                for row in conds_df.itertuples(index=False):
                    cond_hash = getattr(row, 'condition_hash')
                    stim_type = getattr(row, 'type')
                    safe_hash = cond_hash.replace('/', '_')  # paranoia

                    g_c = cg.create_group(safe_hash)
                    g_c.attrs['session'] = s
                    g_c.attrs['scan_idx'] = sc
                    g_c.attrs['type'] = stim_type
                    g_c.attrs['condition_hash'] = cond_hash

                    # JSON snapshot
                    info_dict = {k: getattr(row, k) for k in conds_df.columns
                                 if k not in ('session','scan_idx','condition_hash','type')}
                    j = json.dumps(_to_jsonable(info_dict), ensure_ascii=False)
                    g_c.create_dataset('info_json', data=np.array(j, dtype=object), dtype=vlen_str)

                    # Also emit array-like fields as native datasets when possible
                    for k, v in info_dict.items():
                        try:
                            if isinstance(v, (list, tuple, np.ndarray)):
                                arr = np.asarray(v)
                                if arr.dtype.kind in ('U','S','O'):
                                    g_c.create_dataset(k, data=np.array(arr.astype(str).tolist(), dtype=object), dtype=vlen_str)
                                else:
                                    g_c.create_dataset(k, data=arr)
                            elif isinstance(v, (str, bytes)):
                                g_c.create_dataset(k, data=np.array(str(v), dtype=object), dtype=vlen_str)
                            elif np.isscalar(v):
                                g_c.create_dataset(k, data=np.array(v))
                        except Exception:
                            pass

                    n_conds += 1

            n_scans += 1

        print(f"[save_scans_to_h5] scans saved: {n_scans}, condition entries: {n_conds}")


In [ ]:
# FULL RUN - every scan with a trial table (~19 scans).
scan_keys = (nda.Trial).proj('session', 'scan_idx').fetch(as_dict=True)
save_scans_to_h5('microns_per_scan_stimuli.h5', scan_keys)
